# Chapter 6 — Robot Perception: Measurement Models

**Beam model mixture:**
$$p(z|x,m) = z_{hit}\,p_{hit} + z_{short}\,p_{short} + z_{max}\,p_{max} + z_{rand}\,p_{rand}$$

Each component models a *different physical failure mode*:
- **p_hit**: correct reading (Gaussian around z*)
- **p_short**: closer than expected (unexplained obstacle)
- **p_max**: maximum range hit (glass wall, dark object)
- **p_rand**: uniform noise (multi-path, electronics)

**Likelihood field** is an alternative: pre-compute distance transform on the map,
evaluate sensor endpoints directly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import matplotlib.pyplot as plt
from pluto_filters.pluto_filters.measurement_models.beam_model import BeamModel
from pluto_filters.pluto_filters.measurement_models.likelihood_field import LikelihoodField

print("Imports OK")

In [ ]:
# ── Beam model: 4 components ─────────────────────────────────────────────
z_star   = 3.0   # expected range
z_max_val = 8.0
sigma_hit = 0.5

z_vals = np.linspace(0, z_max_val, 500)

def p_hit(z, z_star, sigma):
    """Gaussian truncated to [0, z_max]""    p = np.exp(-(z - z_star)**2 / (2*sigma**2)) / (sigma * np.sqrt(2*np.pi))
    p[z < 0] = 0; p[z > z_max_val] = 0
    return p / (p.sum() * (z_vals[1]-z_vals[0]) + 1e-10)

def p_short(z, z_star, lam=1.0):
    """Exponential for unexpected nearby obstacles""    p = lam * np.exp(-lam * z)
    p[z > z_star] = 0
    return p / (p.sum() * (z_vals[1]-z_vals[0]) + 1e-10)

ph = p_hit(z_vals, z_star, sigma_hit)
ps = p_short(z_vals, z_star)
pm = np.zeros_like(z_vals); pm[-1] = 1.0  # point mass at z_max
pr = np.ones_like(z_vals) / z_max_val     # uniform

# Mixture weights
w = dict(hit=0.7, short=0.1, max=0.1, rand=0.1)
mixture = w['hit']*ph + w['short']*ps + w['max']*pm + w['rand']*pr

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Beam Model — 4 Failure Modes', fontsize=13)
for ax, (comp, label, color) in zip(axes[:4], [
    (ph, 'p_hit\n(correct reading)', 'steelblue'),
    (ps, 'p_short\n(unexpected obstacle)', 'orange'),
    (pm, 'p_max\n(glass/dark object)', 'gray'),
    (pr, 'p_rand\n(electronics/multipath)', 'pink'),
]):
    ax.fill_between(z_vals, comp, alpha=0.7, color=color)
    ax.axvline(z_star, color='red', ls='--', label=f'z*={z_star}m')
    ax.set_title(label, fontsize=9); ax.set_xlabel('z [m]'); ax.legend(fontsize=7)

axes[4].fill_between(z_vals, mixture, alpha=0.8, color='purple')
axes[4].axvline(z_star, color='red', ls='--', label='z*')
axes[4].set_title(f'Mixture\n(w_hit={w["hit"]}, w_sh={w["short"]}\nw_mx={w["max"]}, w_rd={w["rand"]})', fontsize=8)
axes[4].set_xlabel('z [m]'); axes[4].legend(fontsize=7)

plt.tight_layout(); plt.savefig('ch06_beam_model.png', dpi=120); plt.show()

## Likelihood Field — Pose-Likelihood Surface

In [ ]:
# ── Likelihood field + pose-likelihood surface ──────────────────────────────
# Build occupancy map: hallway with 3 doors
GRID_RES = 0.05  # m per cell
HALL_W, HALL_H = 10.0, 2.0
occ = np.zeros((int(HALL_H/GRID_RES), int(HALL_W/GRID_RES)), dtype=np.float32)

# Walls
occ[0,  :] = 1.0   # bottom wall
occ[-1, :] = 1.0   # top wall
occ[:,  0] = 1.0   # left wall
occ[:, -1] = 1.0   # right wall

# Door openings (remove wall cells)
door_xs = [2.0, 5.0, 8.0]
door_w  = 0.8
for dx in door_xs:
    c0 = int((dx - door_w/2) / GRID_RES)
    c1 = int((dx + door_w/2) / GRID_RES)
    occ[-1, c0:c1] = 0.0
    occ[0,  c0:c1] = 0.0

lf = LikelihoodField(occ, resolution=GRID_RES, sigma=0.3)

# Simulate a 5-ray scan from pose (2.5, 1.0, 0) — facing right
TRUE_POSE_6 = np.array([2.5, 1.0, 0.0])
scan_angles = np.linspace(-np.pi/4, np.pi/4, 5)
scan_ranges  = []
for angle in scan_angles:
    th = TRUE_POSE_6[2] + angle
    for r in np.arange(0, 10, GRID_RES):
        cx = int((TRUE_POSE_6[0] + r*np.cos(th)) / GRID_RES)
        cy = int((TRUE_POSE_6[1] + r*np.sin(th)) / GRID_RES)
        if 0 <= cy < occ.shape[0] and 0 <= cx < occ.shape[1]:
            if occ[cy, cx] > 0.5:
                scan_ranges.append(r); break
    else:
        scan_ranges.append(9.0)

# Scan over x,y grid, compute log-likelihood at each pose
xs_grid = np.linspace(0.5, 9.5, 40)
ys_grid = np.linspace(0.3, 1.7, 20)
LL = np.zeros((len(ys_grid), len(xs_grid)))

for iy, y in enumerate(ys_grid):
    for ix, x in enumerate(xs_grid):
        pose = np.array([x, y, TRUE_POSE_6[2]])
        ll = 0.0
        for angle, rng in zip(scan_angles, scan_ranges):
            th = pose[2] + angle
            ex = pose[0] + rng * np.cos(th)
            ey = pose[1] + rng * np.sin(th)
            ll += lf.measurement_log_likelihood(np.array([ex, ey]))
        LL[iy, ix] = ll

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.imshow(1.0 - occ, origin='lower', cmap='gray', extent=[0, HALL_W, 0, HALL_H])
for dx in door_xs: ax.axvline(dx, color='brown', ls='--', alpha=0.4)
ax.plot(*TRUE_POSE_6[:2], 'g*', ms=14, label='True pose', zorder=5)
for angle, rng in zip(scan_angles, scan_ranges):
    th = TRUE_POSE_6[2] + angle
    ax.plot([TRUE_POSE_6[0], TRUE_POSE_6[0]+rng*np.cos(th)],
            [TRUE_POSE_6[1], TRUE_POSE_6[1]+rng*np.sin(th)], 'r-', lw=1.5, alpha=0.7)
ax.set_title('Occupancy map + simulated scan'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.legend(fontsize=9)

ax = axes[1]
im = ax.imshow(LL, origin='lower', cmap='hot',
               extent=[xs_grid[0], xs_grid[-1], ys_grid[0], ys_grid[-1]], aspect='auto')
plt.colorbar(im, ax=ax, label='log-likelihood')
ax.plot(*TRUE_POSE_6[:2], 'g*', ms=14, label='True pose', zorder=5)
best_iy, best_ix = np.unravel_index(np.argmax(LL), LL.shape)
ax.plot(xs_grid[best_ix], ys_grid[best_iy], 'b^', ms=12, label=f'MLE ({xs_grid[best_ix]:.2f},{ys_grid[best_iy]:.2f})', zorder=5)
ax.set_title('Pose log-likelihood surface\nBright = high likelihood'); ax.legend(fontsize=8)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout(); plt.savefig('ch06_likelihood_field.png', dpi=120); plt.show()
print(f"True pose: ({TRUE_POSE_6[0]}, {TRUE_POSE_6[1]})")
print(f"MLE pose : ({xs_grid[best_ix]:.2f}, {ys_grid[best_iy]:.2f})")

## EM Algorithm — Fit Beam Model Parameters from Data

In [ ]:
# EM: estimate beam model weights from a set of range observations
# Assume we have observed ranges z and we know z* (expected ranges)

np.random.seed(42)

# Simulate 1000 observations with known mixture
TRUE_W = np.array([0.70, 0.10, 0.10, 0.10])
N_OBS  = 1000
z_stars_sim = np.full(N_OBS, 3.0)  # all facing same z*=3m

def sample_beam(z_star, weights, N):
    z_hit_r   = z_star + np.random.normal(0, sigma_hit, N)
    z_short_r = np.random.exponential(1.0, N).clip(0, z_star)
    z_max_r   = np.full(N, z_max_val)
    z_rand_r  = np.random.uniform(0, z_max_val, N)
    component = np.random.choice(4, N, p=weights)
    z = np.where(component==0, z_hit_r,
        np.where(component==1, z_short_r,
        np.where(component==2, z_max_r, z_rand_r)))
    return np.clip(z, 0, z_max_val)

z_obs = sample_beam(3.0, TRUE_W, N_OBS)

# EM iterations
w_em = np.array([0.25, 0.25, 0.25, 0.25])  # uniform init

for iteration in range(50):
    # E-step: compute responsibilities
    eps = 1e-300
    ph_  = p_hit(z_obs, 3.0, sigma_hit) + eps
    ps_  = p_short(z_obs, 3.0) + eps
    pm_  = (np.abs(z_obs - z_max_val) < 0.1).astype(float) * 10 + eps
    pr_  = np.full_like(z_obs, 1.0/z_max_val) + eps
    
    r = np.column_stack([w_em[0]*ph_, w_em[1]*ps_, w_em[2]*pm_, w_em[3]*pr_])
    r = r / r.sum(axis=1, keepdims=True)
    
    # M-step: update weights
    w_em = r.mean(axis=0)

print("=== EM Beam Model Fitting ===")
print(f"True weights   : {TRUE_W}")
print(f"Learned weights: {np.round(w_em, 3)}")
print(f"Error: {np.abs(TRUE_W - w_em).sum():.4f}")

## ✅ Compare | 🔥 Break | 📏 Measure

In [ ]:
# COMPARE: beam model vs likelihood field log-likelihood on same scan
print("=== COMPARE: Beam model vs Likelihood field ===")

# Beam model log-likelihood at true pose
beam_ll_true = sum(
    np.log(w['hit'] * np.interp(r, z_vals, ph) +
           w['short'] * np.interp(r, z_vals, ps) +
           w['max'] * (1 if abs(r-z_max_val)<0.3 else 0) +
           w['rand'] / z_max_val + 1e-300)
    for r in scan_ranges
)

# Likelihood field at true vs offset pose
def lf_loglik(pose):
    ll = 0.0
    for angle, rng in zip(scan_angles, scan_ranges):
        th = pose[2] + angle
        ex = pose[0] + rng * np.cos(th)
        ey = pose[1] + rng * np.sin(th)
        ll += lf.measurement_log_likelihood(np.array([ex, ey]))
    return ll

lf_ll_true   = lf_loglik(TRUE_POSE_6)
lf_ll_offset = lf_loglik(TRUE_POSE_6 + np.array([0.5, 0.3, 0.0]))

print(f"  Beam model log-lik at true pose    : {beam_ll_true:.3f}")
print(f"  LikelihoodField log-lik at true    : {lf_ll_true:.3f}")
print(f"  LikelihoodField log-lik at +0.5m   : {lf_ll_offset:.3f}  (should be lower)\n")

# BREAK: sensor that always returns z_max
print("=== BREAK: degraded sensor (always returns max range) ===")
scan_degraded = [z_max_val] * len(scan_ranges)
ll_degraded = 0.0
for rng in scan_degraded:
    ll_degraded += np.log(w['hit'] * np.interp(rng, z_vals, ph) + w['max']*1.0/0.1 + 1e-300)
print(f"  Degraded scan log-lik: {ll_degraded:.3f}  (much lower than {beam_ll_true:.3f})")
print("  The beam model correctly penalizes a sensor stuck at max range.\n")

# MEASURE
print("=== MEASURE: benchmark over 10 nearby poses ===")
print(f"{'Pose offset':>20}  {'Beam log-lik':>14}  {'LF log-lik':>12}")
print("-" * 52)
for dx in [-0.5, -0.2, 0.0, 0.2, 0.5]:
    test_pose = TRUE_POSE_6 + np.array([dx, 0, 0])
    # Beam model needs scan relative to pose
    scan_test = []
    for angle in scan_angles:
        th = test_pose[2] + angle
        for r in np.arange(0, 10, GRID_RES):
            cx_ = int((test_pose[0] + r*np.cos(th)) / GRID_RES)
            cy_ = int((test_pose[1] + r*np.sin(th)) / GRID_RES)
            if 0 <= cy_ < occ.shape[0] and 0 <= cx_ < occ.shape[1]:
                if occ[cy_, cx_] > 0.5:
                    scan_test.append(r); break
        else:
            scan_test.append(9.0)
    b_ll = sum(np.log(w['hit']*np.interp(r, z_vals, ph)+w['rand']/z_max_val+1e-300) for r in scan_test)
    l_ll = lf_loglik(test_pose)
    marker = '←TRUE' if dx == 0 else ''
    print(f"  x+{dx:>4.1f} {marker:>8}  {b_ll:>14.3f}  {l_ll:>12.3f}")